In [0]:
STORAGE_ACCOUNT  = "storageaccountfull"
BRONZE_CONTAINER = "bronze"
BRONZE_PATH      = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

CATALOG          = "e-commerce"
SILVER_DB        = "silver"
TARGET_TABLE     = "dim_calendar"

In [0]:
import warnings
import math
warnings.filterwarnings("ignore", message=".*No Partition Defined for Window operation.*")

from pyspark.sql.functions import (
    col, date_format, when, lit, concat,
    current_timestamp, year, month, dayofmonth,
    weekofyear, quarter, dayofweek
)
from pyspark.sql.types import IntegerType, DateType

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DB}")
print("Imports ready | Database ensured")

In [0]:
df_bronze = (
    spark.read
    .format("parquet")
    .load(f"{BRONZE_PATH}/calendar")
    .withColumn("source_file", col("_metadata.file_path"))
)

print(f"Bronze rows : {df_bronze.count():,}")
df_bronze.printSchema()
df_bronze.show(5, truncate=False)

In [0]:
df_enriched = (
    df_bronze
    .filter(col("date").isNotNull())
    .withColumn("date",        col("date").cast(DateType()))
    .withColumn("year",        col("year").cast(IntegerType()))
    .withColumn("month",       col("month").cast(IntegerType()))
    .withColumn("day",         col("day").cast(IntegerType()))
    .withColumn("week",        col("week").cast(IntegerType()))
    .withColumn("day_of_week", col("day_of_week").cast(IntegerType()))
    .withColumn("date_sk",
        (col("year") * 10000 + col("month") * 100 + col("day")).cast(IntegerType())
    )
    .withColumn("quarter",
        when(col("month").between(1, 3),  lit(1))
       .when(col("month").between(4, 6),  lit(2))
       .when(col("month").between(7, 9),  lit(3))
       .otherwise(lit(4))
    )
    .withColumn("quarter_name",
        when(col("quarter") == 1, lit("Q1"))
       .when(col("quarter") == 2, lit("Q2"))
       .when(col("quarter") == 3, lit("Q3"))
       .otherwise(lit("Q4"))
    )
    .withColumn("month_name",  date_format(col("date"), "MMMM"))
    .withColumn("month_short", date_format(col("date"), "MMM"))
    .withColumn("day_name",    date_format(col("date"), "EEEE"))
    .withColumn("day_short",   date_format(col("date"), "EEE"))
    .withColumn("is_weekend",
        when(col("day_of_week").isin(0, 6), lit(1))
       .otherwise(lit(0))
    )
    .withColumn("day_type",
        when(col("day_of_week").isin(0, 6), lit("Weekend"))
       .otherwise(lit("Weekday"))
    )
    .withColumn("year_month", date_format(col("date"), "yyyy-MM"))
)

df_enriched = df_enriched.withColumn(
    "year_quarter",
    concat(col("year").cast("string"), lit("-"), col("quarter_name"))
)

print("Calendar enriched")
df_enriched.printSchema()

In [0]:
before     = df_enriched.count()
df_deduped = df_enriched.dropDuplicates(["date"])
print(f"Before : {before:,}  |  After : {df_deduped.count():,}")

In [0]:
df_dim = (
    df_deduped
    .withColumn("ingestion_time", current_timestamp())
    .select(
        col("date_sk"),
        col("date"),
        col("year"),
        col("quarter"),
        col("quarter_name"),
        col("year_quarter"),
        col("month"),
        col("month_name"),
        col("month_short"),
        col("year_month"),
        col("week"),
        col("day"),
        col("day_name"),
        col("day_short"),
        col("day_of_week"),
        col("is_weekend"),
        col("day_type"),
        col("ingestion_time"),
        col("source_file"),
    )
    .orderBy("date")
)

print("Final schema:")
df_dim.printSchema()
df_dim.show(10)

In [0]:
row_count = df_dim.count()
num_partitions = max(2, math.ceil(row_count / 10000))
print(f"{row_count:,} rows -> {num_partitions} partitions")

(
    df_dim.repartition(num_partitions).write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_DB}.{TARGET_TABLE}")
)

print(f"{SILVER_DB}.{TARGET_TABLE} written")

In [0]:
spark.sql(f"""
    SELECT
        MIN(date)  AS date_start,
        MAX(date)  AS date_end,
        COUNT(*)   AS total_days,
        SUM(is_weekend) AS weekend_days,
        COUNT(*) - SUM(is_weekend) AS weekdays
    FROM {SILVER_DB}.{TARGET_TABLE}
""").show()

spark.sql(f"""
    SELECT year, quarter_name, COUNT(*) AS days
    FROM {SILVER_DB}.{TARGET_TABLE}
    GROUP BY year, quarter_name
    ORDER BY year, quarter_name
""").show()

spark.sql(f"SELECT * FROM {SILVER_DB}.{TARGET_TABLE} LIMIT 5").show(truncate=False)